In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica direta com RandomForestRegressor
+ Aplicação global ponto a ponto
+ Classificação multiclasse com split térmico sem overlap
+ Impressão completa das métricas (sem plots extras)
+ Apenas o gráfico principal no estilo solicitado
Autor: Luiz Eduardo Abdala José
"""

import re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, f1_score
)

warnings.filterwarnings("ignore", category=UserWarning)

# ========= PARÂMETROS =========
ARQ_BASE = "base-completo--.pkl"   
REF_TEMP = 30
FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50
SMOOTH_WIN = 5

RF_COMP_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0
)

RF_CLASSIF_PARAMS = dict(
    n_estimators=400,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)


# ========= FUNÇÕES =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c)
            freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]


def add_extra_features(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    return np.hstack([X, mu, sd, amp])


def add_temp_feature(X_aug, temp_vec):
    return np.hstack([X_aug, np.asarray(temp_vec).reshape(-1, 1)])


def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()
    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode='edge')
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode='valid')
    return smooth[:len(arr)]


def spectral_entropy(x):
    x = np.asarray(x, float)
    p = np.abs(x)**2
    s = p.sum()
    if s <= 0: return 0.0
    p /= s
    return float(-np.sum(p * np.log2(p + 1e-12)))


def calc_curve_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)

    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    corr = float(np.corrcoef(y_true, y_pred)[0,1])

    num = float(np.dot(y_true, y_pred))
    den = float(np.linalg.norm(y_true)*np.linalg.norm(y_pred) + 1e-12)
    sam_deg = float(np.degrees(np.arccos(np.clip(num/den, -1, 1))))

    nrmse = rmse / (y_true.max() - y_true.min() + 1e-12)
    diff = y_true - y_pred
    rmsd = float(np.sqrt(np.mean((diff - diff.mean())**2)))
    ccdm = float(1-corr)

    e_true = np.sum(y_true**2)
    e_pred = np.sum(y_pred**2)
    eo = float(2*np.sum(np.minimum(y_true**2, y_pred**2))/(e_true+e_pred+1e-12))

    ent_true = spectral_entropy(y_true)
    ent_pred = spectral_entropy(y_pred)

    return dict(
        R2=r2, RMSE=rmse, MAE=mae, Corr=corr,
        SAM_deg=sam_deg, NRMSE=nrmse, RMSD=rmsd,
        CCDM=ccdm, EnergyOverlap=eo,
        EntropyTrue=ent_true, EntropyPred=ent_pred,
        EntropyDiff=abs(ent_true-ent_pred)
    )


# ========= SCRIPT =========
if __name__ == "__main__":
    timings = {}

    # 1) CARREGAMENTO
    t0 = time.time()
    df = pd.read_pickle(ARQ_BASE)
    fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
    fhz_khz = fhz / 1e3
    df_sem = df[df["falha"] == 0].copy()
    timings["load"] = time.time() - t0

    print(f"Amostras sem falha: {len(df_sem)} | total: {len(df)}")

    # 2) REFERÊNCIA
    t0 = time.time()
    pool_20 = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
    y_ref = np.median(pool_20, axis=0)
    timings["reference"] = time.time() - t0

    # 3) TREINO RF-Comp
    t0 = time.time()
    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)
    Y_target = y_ref[None,:] - X_sem
    X_aug_sem = add_extra_features(X_sem)
    X_comp_sem = add_temp_feature(X_aug_sem, T_sem)
    rf_comp = RandomForestRegressor(**RF_COMP_PARAMS).fit(X_comp_sem, Y_target)
    timings["train_comp"] = time.time() - t0

    # 4) APLICA COMPENSAÇÃO
    t0 = time.time()
    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)
    X_aug_all = add_extra_features(X_all)
    X_comp_all = add_temp_feature(X_aug_all, T_all)
    Y_hat = X_all + rf_comp.predict(X_comp_all)
    for i in range(len(Y_hat)):
        Y_hat[i] = moving_average(Y_hat[i], SMOOTH_WIN)

    df_comp = df.copy()
    df_comp[fcols] = Y_hat
    timings["apply_comp"] = time.time() - t0

    # 5) SPLIT SEM OVERLAP
    t0 = time.time()
    temps = sorted(df["temperatura_c"].unique())
    temps_train = temps[::2]
    temps_test  = temps[1::2]

    df_train = df_comp[df_comp["temperatura_c"].isin(temps_train)]
    df_test  = df_comp[df_comp["temperatura_c"].isin(temps_test)]

    X_train = df_train[fcols].to_numpy(float)
    y_train = df_train["falha"].to_numpy(int)
    X_test  = df_test[fcols].to_numpy(float)
    y_test  = df_test["falha"].to_numpy(int)
    timings["split"] = time.time() - t0

    # 6) CLASSIFICAÇÃO
    t0 = time.time()
    clf = RandomForestClassifier(**RF_CLASSIF_PARAMS).fit(X_train, y_train)
    timings["train_clf"] = time.time() - t0

    t0 = time.time()
    y_pred = clf.predict(X_test)
    timings["predict_clf"] = time.time() - t0

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")



In [ ]:
# 7) MÉTRICAS DA CURVA EXEMPLO
idx_show = df_test.index[10] if len(df_test)>0 else df_test.index[10]
curve_metrics = calc_curve_metrics(y_ref, df_comp.loc[idx_show, fcols].to_numpy(float))

# ========= PRINTA TODAS AS MÉTRICAS ==========

print("\n============== MÉTRICAS DA COMPENSAÇÃO (curva exemplo) ==============")
for k,v in curve_metrics.items():  # Fixed indentation here - removed extra spaces
    print(f"{k:20s}: {v:.6f}")

print("\n==================== CLASSIFICAÇÃO ====================")
print("Matriz de confusão:")
print(cm)
print(f"\nACC   = {acc:.4f}")
print(f"F1    = {macro_f1:.4f}")
print("\nRelatório completo:")
print(classification_report(y_test, y_pred, digits=4))

print("\n==================== TEMPOS (s) ====================")
for k,v in timings.items():  # Fixed indentation here - removed extra spaces
    print(f"{k:20s}: {v:.4f}")

# ========= GRÁFICO FINAL =============
print("\n🔹 Gerando gráfico de exemplo...")

plt.rcParams.update({'font.size': 20,
                     'text.usetex': False,
                     'font.family': "Times New Roman"})

plt.figure(figsize=(12,6))

plt.plot(fhz_khz, y_ref, '--', c='black', lw=0.5,
         label=f"Referência {REF_TEMP}°C")
plt.plot(fhz_khz, df.loc[idx_show, fcols], c='tab:red', alpha=0.6,
         label=f"Original {df.loc[idx_show,'temperatura_c']}°C")
plt.plot(fhz_khz, df_comp.loc[idx_show, fcols], c='tab:blue', lw=2,
         label=f"Compensado RF {df.loc[idx_show,'temperatura_c']}°C")

plt.title(f"Random Forest — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.legend()
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

In [ ]:
print("\n===== ÍNDICES DISPONÍVEIS EM df_test =====")

for i, idx in enumerate(df_test.index):
    temp = df_test.loc[idx, "temperatura_c"]
    print(f"Posição {i:3d}  |  Índice real: {idx}  |  Temperatura: {temp}°C")

In [ ]:
# 7) MÉTRICAS DA CURVA EXEMPLO
idx_show = df_test.index[17] if len(df_test)>0 else df_test.index[17]
curve_metrics = calc_curve_metrics(y_ref, df_comp.loc[idx_show, fcols].to_numpy(float))

# ========= PRINTA TODAS AS MÉTRICAS ==========

print("\n============== MÉTRICAS DA COMPENSAÇÃO (curva exemplo) ==============")
for k,v in curve_metrics.items():  # Fixed indentation here - removed extra spaces
    print(f"{k:20s}: {v:.6f}")

print("\n==================== CLASSIFICAÇÃO ====================")
print("Matriz de confusão:")
print(cm)
print(f"\nACC   = {acc:.4f}")
print(f"F1    = {macro_f1:.4f}")
print("\nRelatório completo:")
print(classification_report(y_test, y_pred, digits=4))

print("\n==================== TEMPOS (s) ====================")
for k,v in timings.items():  # Fixed indentation here - removed extra spaces
    print(f"{k:20s}: {v:.4f}")

# ========= GRÁFICO FINAL =============
print("\n🔹 Gerando gráfico de exemplo...")

plt.rcParams.update({'font.size': 10,
                     'text.usetex': False,
                     'font.family': "Times New Roman"})

plt.figure(figsize=(8,4))

plt.plot(fhz_khz, y_ref, '--', c='black', lw=0.5,
         label=f"Referência {REF_TEMP}°C")
plt.plot(fhz_khz, df.loc[idx_show, fcols], c='tab:red', alpha=0.6,
         label=f"Original {df.loc[idx_show,'temperatura_c']}°C")
plt.plot(fhz_khz, df_comp.loc[idx_show, fcols], c='tab:blue', lw=2,
         label=f"Compensado RF {df.loc[idx_show,'temperatura_c']}°C")

plt.title(f"Random Forest — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.legend(frameon=True,
           facecolor='white',
           edgecolor='none')
plt.grid(alpha=0.0)
plt.tight_layout()
plt.show()